In [2]:
import sqlite3
import time
import re
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import xml.etree.ElementTree as ET

DB_NAME = "weworkremotely_jobs.db"
BASE_URL = "https://weworkremotely.com"

# 主要なカテゴリページ
CATEGORY_URLS = [
    "/categories/remote-programming-jobs",
    "/categories/remote-devops-sysadmin-jobs",
    "/categories/remote-design-jobs",
    "/categories/remote-product-jobs",
    "/categories/remote-marketing-jobs",
    "/categories/remote-customer-support-jobs",
    "/categories/remote-sales-jobs",
    "/categories/remote-writing-jobs",
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def extract_job_listings_from_rss(rss_text, category):
    """RSSフィードからジョブ情報を抽出"""
    jobs = []
    
    try:
        # XMLをパース
        root = ET.fromstring(rss_text)
        
        # RSSの名前空間を処理
        namespaces = {
            'media': 'http://search.yahoo.com/mrss',
            'dc': 'http://purl.org/dc/elements/1.1/'
        }
        
        # すべてのitemを取得
        items = root.findall('.//item')
        
        print(f"  {len(items)} 件の求人要素を発見")
        
        for item in items:
            try:
                # タイトルを取得
                title_elem = item.find('title')
                title = title_elem.text if title_elem is not None else ""
                
                # URLを取得
                link_elem = item.find('link')
                job_url = link_elem.text if link_elem is not None else ""
                
                # 会社名を抽出（タイトルから）
                company = ""
                if title and ":" in title:
                    parts = title.split(":", 1)
                    company = parts[0].strip()
                    title = parts[1].strip() if len(parts) > 1 else title
                
                # 地域を取得
                region_elem = item.find('region')
                region = region_elem.text if region_elem is not None else "Remote"
                
                # カテゴリを取得
                category_elem = item.find('category')
                job_category = category_elem.text if category_elem is not None else category
                
                if not job_url or not title:
                    continue
                
                jobs.append({
                    "title": title,
                    "company": company,
                    "category": job_category,
                    "region": region,
                    "url": job_url,
                })
                
            except Exception as e:
                print(f"    ジョブアイテムの解析エラー: {e}")
                continue
        
    except ET.ParseError as e:
        print(f"  XML解析エラー: {e}")
    except Exception as e:
        print(f"  予期しないエラー: {e}")
    
    return jobs

def scrape_job_detail(job_url):
    """個別の求人詳細ページから情報を取得"""
    try:
        print(f"    詳細取得: {job_url}")
        response = requests.get(job_url, headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # 求人詳細を取得
        description_tag = soup.select_one(".listing-container, #job-listing-show-container, .job-description")
        description = description_tag.get_text(strip=True, separator=" ") if description_tag else ""
        
        # ジョブタイプ
        job_type_tag = soup.select_one(".job-type, .listing-tag")
        job_type = job_type_tag.get_text(strip=True) if job_type_tag else "Full-time"
        
        # 投稿日
        posted_tag = soup.select_one("time, .posted-date")
        posted_date = ""
        if posted_tag:
            posted_date = posted_tag.get("datetime", "") or posted_tag.get_text(strip=True)
        
        # 給与情報（説明文から抽出）
        salary_patterns = [
            r'\$[\d,]+\s*[-–]\s*\$?[\d,]+[kK]?',
            r'\$[\d,]+[kK]?',
            r'€[\d,]+\s*[-–]\s*€?[\d,]+[kK]?',
            r'£[\d,]+\s*[-–]\s*£?[\d,]+[kK]?'
        ]
        salary_info = ""
        for pattern in salary_patterns:
            match = re.search(pattern, description)
            if match:
                salary_info = match.group(0)
                break
        
        return {
            "description": description,
            "job_type": job_type,
            "posted_date": posted_date,
            "salary_info": salary_info,
        }
    except Exception as e:
        print(f"    詳細取得エラー: {e}")
        return {
            "description": "",
            "job_type": "Full-time",
            "posted_date": "",
            "salary_info": "",
        }

def analyze_job(title, description):
    """求人のタイトルと説明文を分析してフラグを設定"""
    title_lower = title.lower()
    desc_lower = description.lower()
    combined = title_lower + " " + desc_lower
    
    # 福利厚生の検出（より広範なキーワード）
    has_macbook = any(keyword in combined for keyword in [
        "macbook", "mac book", "apple laptop", "laptop provided",
        "company laptop", "equipment provided", "work equipment"
    ])
    
    has_company_retreat = any(keyword in combined for keyword in [
        "company retreat", "team retreat", "annual retreat", 
        "offsites", "team offsite", "company trip", "team gathering",
        "in-person gathering", "team meetup", "yearly gathering"
    ])
    
    has_unlimited_pto = any(keyword in combined for keyword in [
        "unlimited pto", "unlimited vacation", "unlimited time off",
        "unlimited paid time off", "flexible vacation", "unlimited leave",
        "flexible time off", "unlimited days off"
    ])
    
    # 包括的な福利厚生
    has_premium_benefits = has_macbook or has_company_retreat or has_unlimited_pto
    
    # タイトル分析（職位レベル）
    is_senior = any(keyword in title_lower for keyword in [
        "senior", "sr.", "sr ", "principal", "staff", "lead"
    ])
    
    is_lead = any(keyword in title_lower for keyword in [
        "lead", "head of", "director", "manager", "vp", "chief", "team lead"
    ])
    
    is_junior = any(keyword in title_lower for keyword in [
        "junior", "jr.", "jr ", "associate", "early career"
    ])
    
    is_entry = any(keyword in title_lower for keyword in [
        "entry", "entry-level", "entry level", "intern", "graduate", "trainee"
    ])
    
    return {
        "has_macbook": has_macbook,
        "has_company_retreat": has_company_retreat,
        "has_unlimited_pto": has_unlimited_pto,
        "has_premium_benefits": has_premium_benefits,
        "is_senior": is_senior,
        "is_lead": is_lead,
        "is_junior": is_junior,
        "is_entry": is_entry,
    }

def scrape_category(category_url, max_jobs=30):
    """特定のカテゴリから求人を取得"""
    jobs = []
    category_name = category_url.split("/")[-1].replace("remote-", "").replace("-jobs", "")
    
    try:
        url = BASE_URL + category_url
        print(f"\nスクレイピング中: {url}")
        
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        # レスポンスがRSSフィードかどうか確認
        if response.text.strip().startswith('<?xml') or '<rss' in response.text[:200]:
            print(f"  RSSフィード形式を検出")
            page_jobs = extract_job_listings_from_rss(response.text, category_name)
        else:
            print(f"  HTML形式を検出（フォールバック）")
            soup = BeautifulSoup(response.text, "html.parser")
            page_jobs = []  # HTMLパース用の関数を後で追加可能
        
        if not page_jobs:
            print(f"  求人が見つかりませんでした")
            return jobs
        
        print(f"  {len(page_jobs)} 件の求人を発見")
        
        # 最大数まで取得
        for job in page_jobs[:max_jobs]:
            jobs.append(job)
        
        # 丁寧な待機時間
        time.sleep(3)
        
    except Exception as e:
        print(f"カテゴリスクレイピングエラー: {e}")
    
    return jobs

def save_to_db(jobs):
    """求人データをデータベースに保存"""
    conn = sqlite3.connect(DB_NAME)
    cur = conn.cursor()
    
    saved_count = 0
    skipped_count = 0
    error_count = 0
    
    for i, job in enumerate(jobs, 1):
        try:
            print(f"\n[{i}/{len(jobs)}] 処理中: {job['title'][:50]}...")
            
            # 詳細情報を取得
            if job["url"]:
                detail = scrape_job_detail(job["url"])
                time.sleep(2)  # 丁寧な待機
            else:
                detail = {
                    "description": "",
                    "job_type": "Full-time",
                    "posted_date": "",
                    "salary_info": "",
                }
            
            # 分析フラグを取得
            analysis = analyze_job(job["title"], detail["description"])
            
            # データベースに挿入
            cur.execute("""
                INSERT OR IGNORE INTO jobs (
                    title, company, category, job_type, location,
                    posted_date, url, description, salary_info,
                    has_macbook, has_company_retreat, has_unlimited_pto, 
                    has_premium_benefits, is_senior, is_lead, is_junior, is_entry
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                job["title"],
                job["company"],
                job["category"],
                detail["job_type"],
                job["region"],
                detail["posted_date"],
                job["url"],
                detail["description"],
                detail["salary_info"],
                analysis["has_macbook"],
                analysis["has_company_retreat"],
                analysis["has_unlimited_pto"],
                analysis["has_premium_benefits"],
                analysis["is_senior"],
                analysis["is_lead"],
                analysis["is_junior"],
                analysis["is_entry"],
            ))
            
            if cur.rowcount > 0:
                saved_count += 1
                print(f"  ✓ 保存完了")
            else:
                skipped_count += 1
                print(f"  - スキップ（重複）")
            
            # 定期的にコミット
            if i % 10 == 0:
                conn.commit()
                print(f"\n--- 進捗: {saved_count}件保存, {skipped_count}件スキップ ---")
                
        except Exception as e:
            error_count += 1
            print(f"  ✗ エラー: {e}")
            continue
    
    conn.commit()
    conn.close()
    
    print(f"\n{'='*60}")
    print(f"保存完了: {saved_count} 件の新規求人")
    print(f"スキップ: {skipped_count} 件の重複求人")
    print(f"エラー: {error_count} 件")
    print(f"{'='*60}")

def main():
    """メイン実行関数"""
    print("=" * 60)
    print("We Work Remotely 求人スクレイピング開始")
    print("=" * 60)
    print(f"開始時刻: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    all_jobs = []
    
    # 各カテゴリから求人を取得
    for category_url in CATEGORY_URLS:
        print(f"\n{'='*60}")
        print(f"カテゴリ: {category_url}")
        print(f"{'='*60}")
        
        jobs = scrape_category(category_url, max_jobs=20)
        all_jobs.extend(jobs)
        
        print(f"\nこのカテゴリで {len(jobs)} 件取得")
        print(f"累計: {len(all_jobs)} 件")
        
        # カテゴリ間の待機
        time.sleep(5)
    
    print(f"\n{'='*60}")
    print(f"総取得数: {len(all_jobs)} 件")
    print(f"{'='*60}")
    
    # 重複URLを削除
    unique_jobs = []
    seen_urls = set()
    for job in all_jobs:
        if job["url"] not in seen_urls and job["url"]:
            unique_jobs.append(job)
            seen_urls.add(job["url"])
    
    print(f"重複削除後: {len(unique_jobs)} 件")
    
    # データベースに保存
    if unique_jobs:
        print(f"\n{'='*60}")
        print("データベースに保存中...")
        print(f"{'='*60}")
        save_to_db(unique_jobs)
    else:
        print("\n警告: 取得できた求人がありません")
    
    print(f"\n{'='*60}")
    print("スクレイピング完了")
    print(f"終了時刻: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}")

if __name__ == "__main__":
    main()

We Work Remotely 求人スクレイピング開始
開始時刻: 2026-01-27 21:05:15

カテゴリ: /categories/remote-programming-jobs

スクレイピング中: https://weworkremotely.com/categories/remote-programming-jobs
  RSSフィード形式を検出
  25 件の求人要素を発見
  25 件の求人を発見

このカテゴリで 20 件取得
累計: 20 件

カテゴリ: /categories/remote-devops-sysadmin-jobs

スクレイピング中: https://weworkremotely.com/categories/remote-devops-sysadmin-jobs
  HTML形式を検出（フォールバック）
  求人が見つかりませんでした

このカテゴリで 0 件取得
累計: 20 件

カテゴリ: /categories/remote-design-jobs

スクレイピング中: https://weworkremotely.com/categories/remote-design-jobs
  HTML形式を検出（フォールバック）
  求人が見つかりませんでした

このカテゴリで 0 件取得
累計: 20 件

カテゴリ: /categories/remote-product-jobs

スクレイピング中: https://weworkremotely.com/categories/remote-product-jobs
  HTML形式を検出（フォールバック）
  求人が見つかりませんでした

このカテゴリで 0 件取得
累計: 20 件

カテゴリ: /categories/remote-marketing-jobs

スクレイピング中: https://weworkremotely.com/categories/remote-marketing-jobs
  HTML形式を検出（フォールバック）
  求人が見つかりませんでした

このカテゴリで 0 件取得
累計: 20 件

カテゴリ: /categories/remote-customer-support-jobs

スクレイピング中: https://

In [3]:
import sqlite3
import pandas as pd

# データベースに接続
conn = sqlite3.connect("weworkremotely_jobs.db")

# データを取得
df = pd.read_sql_query("SELECT * FROM jobs", conn)

# 基本情報を表示
print(f"総求人数: {len(df)}")
print(f"\nカラム: {list(df.columns)}")
print(f"\n最初の5件:")
print(df.head())

# カテゴリ別の件数
print(f"\n\nカテゴリ別件数:")
print(df['category'].value_counts())

conn.close()

総求人数: 114

カラム: ['id', 'title', 'company', 'category', 'job_type', 'location', 'posted_date', 'url', 'description', 'has_macbook', 'has_company_retreat', 'has_unlimited_pto', 'has_premium_benefits', 'is_senior', 'is_lead', 'is_junior', 'is_entry', 'salary_info', 'scraped_at']

最初の5件:
   id                        title     company     category   job_type  \
0  66     Senior Software Engineer    TechCorp  Programming  Full-time   
1  67  Senior Software Engineer #2  TechCorp 2  Programming  Full-time   
2  68  Senior Software Engineer #3  TechCorp 3  Programming  Full-time   
3  69  Senior Software Engineer #4  TechCorp 4  Programming  Full-time   
4  70  Senior Software Engineer #5  TechCorp 5  Programming  Full-time   

  location posted_date                                                url  \
0   Remote  2025-12-06  https://weworkremotely.com/remote-jobs/test-jo...   
1   Remote  2026-01-19  https://weworkremotely.com/remote-jobs/test-jo...   
2   Remote  2026-01-22  https://weworkr